# Reconocimiento de flores por imagen - Oxford Flowers-102

**Que hace este notebook.** Recorre el proyecto de principio a fin: comprueba el
entorno de Colab, obtiene el codigo, entrena una red (*transfer learning* sobre
EfficientNet-B0) para clasificar 102 especies de flores a partir de una foto, mide
el resultado sobre el split oficial de test y lanza una demo interactiva.

**Pipeline**

1. Comprobar el entorno e instalar SOLO lo que falte (nunca `torch`).
2. Montar Google Drive (opcional) y restaurar el dataset cacheado.
3. Obtener el codigo del proyecto (modo `auto`: proyecto local o clonado de GitHub).
4. Imports del proyecto.
5. Configurar la corrida (modo debug o modo completo).
6. Entrenar y guardar el checkpoint.
7. Evaluar sobre el split oficial de test (top-1, top-5, macro-F1, matriz de confusion).
8. Imprimir el resumen listo para pegar en el informe.
9. Demo Gradio: subes una foto y recibes el top-5 de especies con su probabilidad.
10. Empaquetar y descargar `resultados/` en un ZIP + cachear `data/` en Drive.

**Que necesitas**

- Una cuenta de Google (Colab y Drive son gratuitos).
- Acelerador **GPU T4**: `Entorno de ejecucion > Cambiar tipo de entorno de ejecucion > T4 GPU`.
- Conexion a internet: el notebook clona el repositorio
  <https://github.com/Luis92561/I.A-Flores> por ti. Si prefieres no clonar, tambien
  puedes subir el proyecto como `IA_flores_colab.zip` (lo genera
  `python empaquetar_colab.py`).

**Como usarlo.** Ejecuta las celdas EN ORDEN; `Entorno de ejecucion > Ejecutar todo`
tambien vale si ya elegiste la GPU. El **modo debug** viene activado por defecto: sirve
para validar la instalacion completa en 1-2 minutos antes de gastar GPU en la corrida
completa (25 epocas, ~15-25 min en T4).

**Estado.** Los modulos `src/data.py`, `src/model.py`, `src/train.py`, `src/evaluate.py`,
`src/predict.py` y `src/app.py` estan terminados y verificados con sus tests
(17/17, 22/22 y el resto), y el pipeline completo se probo de punta a punta
(entrenar -> evaluar -> predecir). El unico paso que falta es esta corrida en Colab.

## Paso 1 - Entorno: GPU y dependencias

**GPU.** El entrenamiento completo necesita GPU. En Colab se activa en
`Entorno de ejecucion > Cambiar tipo de entorno de ejecucion > Acelerador de hardware: T4 GPU`.
Si `torch.cuda.is_available()` es `False` el notebook sigue funcionando en CPU (mucho mas
lento), pero entrenar 25 epocas en CPU no es razonable.

**Regla de oro de Colab.** *Nunca* se ejecuta `pip install -r requirements.txt` aqui:
Colab ya trae `torch` y `torchvision` compilados contra CUDA y reinstalarlos por pip puede
sustituirlos por ruedas **CPU** y dejar la T4 inservible. Esta celda:

1. **Comprueba** `torch` / `torchvision` / `scipy` y **no los instala nunca**.
2. Instala por `pip install -q` **solo lo que falte** de `gradio`, `scikit-learn`,
   `matplotlib` y `scipy`.
3. Informa de las dependencias de apoyo (`numpy`, `pandas`, `Pillow`, `tqdm`).
4. Imprime un resumen de versiones (torch, torchvision, CUDA y GPU).

In [ ]:
import importlib
import subprocess
import sys


def importable(nombre):
    # True si el modulo se puede importar en este kernel.
    try:
        importlib.import_module(nombre)
        return True
    except Exception:
        return False


# --- 1.1 torch / torchvision: SOLO se comprueban, jamas se instalan ---------
TORCH_OK = importable("torch") and importable("torchvision")
if TORCH_OK:
    print("torch y torchvision ya estan disponibles: no se tocan (regla de Colab).")
else:
    print("AVISO: falta torch y/o torchvision en este entorno.")
    print("  - En Colab no deberia pasar: revisa 'Entorno de ejecucion > T4 GPU'.")
    print("  - Fuera de Colab instalalos tu a proposito: pip install torch torchvision")
    print("  (aqui NO se hace: en Colab romperia la GPU.)")

# --- 1.2 paquetes que Colab no garantiza: pip SOLO si faltan ---------------
REQUERIDOS = {
    "gradio": "gradio",           # demo interactivo (paso 9)
    "sklearn": "scikit-learn",    # metricas y matriz de confusion (paso 7)
    "matplotlib": "matplotlib",   # figuras de curvas y confusion
    "scipy": "scipy",             # Flowers102 lee los .mat con scipy.io
}
faltantes = [pip_name for modulo, pip_name in REQUERIDOS.items() if not importable(modulo)]

if faltantes:
    print("Instalando por pip (solo lo que falta):", faltantes)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *faltantes])
else:
    print("Todo lo necesario ya estaba instalado: no se llama a pip.")

# --- 1.3 paquetes de apoyo (informativo) -----------------------------------
print("\nDependencias de apoyo:")
for modulo in ("numpy", "pandas", "PIL", "tqdm"):
    print("  %-8s: %s" % (modulo, "OK" if importable(modulo) else "FALTA"))

# --- 1.4 resumen de versiones ----------------------------------------------
print("\n=== Resumen del entorno ===")
try:
    import torch
    import torchvision

    print("torch          :", torch.__version__)
    print("torchvision    :", torchvision.__version__)
    hay_cuda = torch.cuda.is_available()
    print("CUDA disponible:", hay_cuda)
    if hay_cuda:
        props = torch.cuda.get_device_properties(0)
        print("CUDA runtime   :", torch.version.cuda)
        print("GPU            :", torch.cuda.get_device_name(0))
        print("memoria        : %.1f GB" % (props.total_memory / 1024 ** 3))
    else:
        print("AVISO: no hay GPU. Entorno de ejecucion > Cambiar tipo de")
        print("       entorno de ejecucion > T4 GPU > Guardar. El notebook")
        print("       funciona en CPU, pero la corrida completa tarda muchisimo mas.")
except ImportError:
    print("torch no importable -> el entrenamiento fallara. Arreglalo antes de seguir.")
print("===========================")

## Paso 2 - Google Drive (opcional) y cache del dataset

**Por que Drive.** Colab borra el disco al desconectarse. Si montas Drive, el checkpoint
(`best.pt`), las metricas y las figuras se copian a `MyDrive/IA_flores` y sobreviven a una
desconexion. Pon `USAR_DRIVE = False` si prefieres no usar Drive.

**Por que cachear `data/`.** El dataset Flowers-102 pesa ~330 MB y se descarga en cada
sesion nueva de Colab. Esta celda lo **restaura desde Drive** al empezar y la ultima celda
del notebook lo **copia a Drive** al terminar: asi solo se descarga una vez.

- Si **no** hay Drive (fuera de Colab, o `USAR_DRIVE = False`), no pasa nada: la celda
  avisa y continua.
- Todo esta envuelto en `try/except`: si Drive falla, el notebook sigue igual.
- Rutas: `data/` (dataset), `resultados/` (checkpoint, metricas, figuras, predicciones).

In [ ]:
import shutil
import sys
from pathlib import Path

# --------------------------------------------------------------- ajustes ----
USAR_DRIVE = True      # ponlo en False si no quieres persistir nada en Drive
DRIVE_DIR = None       # se rellena si Drive queda montado
EN_COLAB = False       # True si estamos dentro de Google Colab

try:
    import google.colab  # noqa: F401  (este modulo solo existe en Colab)

    EN_COLAB = True
except ImportError:
    EN_COLAB = False

# ---------------------------------------------------------------- montaje ---
if EN_COLAB and USAR_DRIVE:
    try:
        from google.colab import drive

        drive.mount("/content/drive")
        DRIVE_DIR = Path("/content/drive/MyDrive/IA_flores")
        DRIVE_DIR.mkdir(parents=True, exist_ok=True)
        print("Drive montado. Carpeta de respaldo:", DRIVE_DIR)
    except Exception as exc:
        DRIVE_DIR = None
        print("No se pudo montar Drive:", exc)
        print("El notebook continua igual; descarga el ZIP del paso 10 antes de cerrar.")
elif EN_COLAB:
    print("USAR_DRIVE=False: los resultados solo viviran durante esta sesion.")
else:
    print("Fuera de Colab (o sin google.colab): se omite el montaje de Drive.")
    print("Nota: los avisos de 'Acceso denegado' al escribir en MyDrive fuera de Colab")
    print("      son normales; el notebook funciona igual en modo local.")

# ------------------------------------------- restaurar dataset desde Drive --
# Solo se copia si data/ no existe todavia: si ya esta, no se toca.
RAIZ_ACTUAL = Path.cwd()
DATA_LOCAL = RAIZ_ACTUAL / "data"

if DRIVE_DIR is None:
    print("\nCache del dataset: sin Drive, se omite la restauracion.")
elif DATA_LOCAL.exists():
    print("\nCache del dataset: ya existe", DATA_LOCAL, "-> no se restaura nada.")
else:
    DATA_DRIVE = DRIVE_DIR / "data"
    try:
        if DATA_DRIVE.is_dir() and any(DATA_DRIVE.iterdir()):
            print("\nRestaurando data/ desde", DATA_DRIVE, "...")
            shutil.copytree(DATA_DRIVE, DATA_LOCAL)
            print("Dataset restaurado: se evita la descarga de ~330 MB.")
        else:
            print("\nCache del dataset: no hay copia en Drive todavia (se descargara).")
    except Exception as exc:
        print("\nNo se pudo restaurar data/ desde Drive:", exc)
        print("Se descargara el dataset otra vez; el notebook continua.")

## Paso 3 - Obtener el codigo del proyecto

`MODO_CODIGO` admite cuatro vias:

| `MODO_CODIGO` | Cuando usarlo | Que hace |
|---|---|---|
| `"auto"` (defecto) | **Siempre** | Si ya encuentra `src/config.py` (cwd, subcarpeta o carpeta padre) usa ese proyecto; si no, **clona el repositorio de GitHub** |
| `"clone"` | Quieres el codigo desde GitHub | `git clone --depth 1` del repositorio en `/content/IA_flores` |
| `"zip"` | No puedes clonar (sin internet o repo privado) | Pide subir `IA_flores_colab.zip`, lo descomprime y usa la carpeta con `src/config.py` |
| `"drive"` | Ya tienes la carpeta del proyecto en Drive | La copia a `/content/IA_flores` |

Repositorio del proyecto: <https://github.com/Luis92561/I.A-Flores>

Internamente `"auto"` se resuelve a `"local"` (proyecto ya presente) o a `"clone"` (lo
descarga de GitHub); es la unica via que se elige sola. Las demas se fuerzan cambiando
`MODO_CODIGO` en la celda de abajo.

Notas:

- **En Colab no necesitas subir nada**: deja `MODO_CODIGO = "auto"` y el notebook clona
  el repositorio solo. Lo unico que hace falta es conexion a internet.
- `"auto"` es tambien lo que permite ejecutar este mismo notebook **fuera de Colab**
  (Jupyter local, VS Code): detecta el proyecto y no descarga nada.
- Para `"zip"`: comprime la **carpeta del proyecto** (con `src/` y `notebooks/` dentro),
  no los archivos sueltos. Puedes generarlo con `python empaquetar_colab.py`.
- La celda termina cambiando el directorio de trabajo a la raiz del proyecto y comprueba
  que existe `src/config.py`; si no lo encuentra, se detiene con un mensaje claro en vez
  de fallar mas adelante.
- Los modulos `src/data.py`, `src/model.py`, `src/train.py`, `src/evaluate.py`,
  `src/predict.py` y `src/app.py` son parte del proyecto: el notebook los importa por su
  API publica.


In [ ]:
import os
import shutil
import zipfile
from pathlib import Path

# Via de obtencion del codigo: "auto" | "zip" | "clone" | "drive"
MODO_CODIGO = "auto"

# En Colab el disco de trabajo es /content; fuera de Colab, el directorio actual.
TRABAJO = "/content" if Path("/content").is_dir() else "."

# Opcion "clone": repositorio del proyecto en GitHub (publico).
REPO_GIT = "https://github.com/Luis92561/I.A-Flores.git"
RAIZ_CLON = TRABAJO + "/IA_flores"

# Opcion "zip": nombre del ZIP que sube el usuario.
NOMBRE_ZIP = "IA_flores_colab.zip"
RUTA_ZIP = TRABAJO + "/" + NOMBRE_ZIP

# Opcion "drive": carpeta del proyecto dentro de MyDrive.
ORIGEN_DRIVE = TRABAJO + "/drive/MyDrive/IA_flores"


def raiz_con_config(base, profundo=True):
    # Devuelve la carpeta que contiene src/config.py, o None si no existe.
    # `profundo=True` anade una busqueda recursiva, util cuando el ZIP trae una
    # carpeta envoltorio. En el recorrido de carpetas padre se usa `profundo=False`
    # a proposito: recorrer todo el arbol de /content (con Drive montado) podria
    # tardar muchisimo y ademas encontraria copias antiguas del proyecto.
    base = Path(base)
    if not base.exists():
        return None
    if (base / "src" / "config.py").is_file():
        return base.resolve()
    for sub in sorted(base.glob("*/src/config.py")):
        return sub.parent.parent.resolve()
    if profundo:
        for sub in sorted(base.rglob("src/config.py")):
            return sub.parent.parent.resolve()
    return None


def detectar_proyecto():
    # Busca el proyecto en el cwd y en sus carpetas padre (caso Jupyter local),
    # sin recorrer el arbol completo de cada candidato.
    actual = Path.cwd().resolve()
    for candidato in [actual, *actual.parents]:
        raiz = raiz_con_config(candidato, profundo=False)
        if raiz is not None:
            return raiz
    return None


raiz = detectar_proyecto()

if MODO_CODIGO == "auto":
    if raiz is not None:
        print("MODO_CODIGO='auto': proyecto detectado en", raiz)
        MODO_CODIGO = "local"
    else:
        print("MODO_CODIGO='auto': no hay proyecto local -> se clona el repositorio.")
        MODO_CODIGO = "clone"

if MODO_CODIGO == "local":
    # (a) El proyecto ya esta aqui (o en una carpeta padre): no hay nada que copiar.
    print("Se usa el proyecto local en", raiz)

elif MODO_CODIGO == "clone":
    # (b) Clonar el repositorio. Si es privado, usa un token en la URL.
    if not REPO_GIT:
        raise ValueError("MODO_CODIGO='clone' pero REPO_GIT esta vacio: pon la URL del repositorio.")
    if Path(RAIZ_CLON).exists():
        print("Ya existe", RAIZ_CLON, "-> no se vuelve a clonar.")
    else:
        os.system('git clone --depth 1 "%s" "%s"' % (REPO_GIT, RAIZ_CLON))
    raiz = raiz_con_config(RAIZ_CLON)

elif MODO_CODIGO == "zip":
    # (c) Subir el ZIP del proyecto desde el PC.
    if not Path(RUTA_ZIP).exists():
        try:
            from google.colab import files   # solo existe en Colab

            print("Selecciona el ZIP del proyecto (" + NOMBRE_ZIP + ")...")
            subidos = files.upload()
            for nombre in subidos:
                if Path(nombre).resolve() != Path(RUTA_ZIP).resolve():
                    shutil.move(nombre, RUTA_ZIP)
        except ImportError:
            print("Fuera de Colab: copia el ZIP a mano en", RUTA_ZIP)

    if not Path(RUTA_ZIP).exists():
        raise FileNotFoundError(
            "No encuentro %s. Sube el ZIP (python empaquetar_colab.py) o usa MODO_CODIGO='clone'."
            % RUTA_ZIP
        )

    extraido = Path(TRABAJO) / "_extraido"
    if extraido.exists():
        shutil.rmtree(extraido)
    with zipfile.ZipFile(RUTA_ZIP) as zf:
        zf.extractall(extraido)
    raiz = raiz_con_config(extraido)
    if raiz is None:
        raise FileNotFoundError(
            "El ZIP no contiene src/config.py. Comprime la carpeta del proyecto, no los archivos sueltos."
        )
    print("ZIP extraido en", raiz)

elif MODO_CODIGO == "drive":
    # (d) Copiar la carpeta del proyecto que ya esta en Drive.
    if not Path(ORIGEN_DRIVE).exists():
        raise FileNotFoundError("No encuentro %s. Sube la carpeta del proyecto a Drive." % ORIGEN_DRIVE)
    destino = Path(TRABAJO) / "IA_flores"
    shutil.copytree(ORIGEN_DRIVE, destino, dirs_exist_ok=True)
    raiz = raiz_con_config(destino)
    print("Proyecto copiado desde Drive a", destino)

else:
    raise ValueError("MODO_CODIGO debe ser 'auto', 'local', 'zip', 'clone' o 'drive'.")

if raiz is None:
    raise FileNotFoundError(
        "No encuentro src/config.py. Revisa la via elegida en MODO_CODIGO (ahora: %r)." % MODO_CODIGO
    )

# La raiz del proyecto pasa a ser el directorio de trabajo: los imports y las rutas
# relativas de Config (data/, resultados/) cuelgan de aqui.
os.chdir(raiz)

assert Path("src/config.py").is_file(), "Falta src/config.py en la raiz del proyecto."
print("Raiz del proyecto:", Path.cwd())
print("src/config.py    : OK")
print("Contenido        :", sorted(x.name for x in Path(".").iterdir()))

## Paso 4 - Imports del proyecto

Anadimos la raiz del proyecto a `sys.path` y cargamos las piezas de la Fase 0
(seccion 3 de `contrato interno`):

- `Config`, `IMAGE_SIZE`, `FLOWERS102_CLASSES` desde `src.config`.
- `set_seed`, `resolve_device`, `describe_device`, `load_json` desde `src.utils`.

`resolve_device("auto")` devuelve CUDA si existe y CPU si no. En el codigo del proyecto no
hay ningun `.cuda()` duro, asi que todo funciona en ambos entornos.

In [ ]:
import json
import sys
import time
from pathlib import Path

RAIZ_PROYECTO = str(Path.cwd())
if RAIZ_PROYECTO not in sys.path:
    sys.path.insert(0, RAIZ_PROYECTO)   # permite "from src.xxx import ..."

import matplotlib.pyplot as plt                                          # noqa: E402

from src.config import Config, FLOWERS102_CLASSES, IMAGE_SIZE            # noqa: E402
from src.utils import describe_device, load_json, resolve_device, set_seed  # noqa: E402

print("Imports del proyecto OK.")
print("Raiz en sys.path :", RAIZ_PROYECTO)
print("Contrato         : img_size = %d px | num_classes = %d" % (IMAGE_SIZE, FLOWERS102_CLASSES))

## Paso 5 - Configuracion: modo debug y modo completo

| | `MODO_DEBUG = True` (por defecto) | `MODO_DEBUG = False` |
|---|---|---|
| Imagenes | subconjunto de `debug_per_class` por clase (102 x 4 x 3) | 1020 train / 1020 val / 6149 test |
| Epocas | 2 | 25 |
| AMP (precision mixta) | desactivado (lo fuerza `Config`) | activado si hay CUDA |
| Salida | `resultados/debug` (no pisa la corrida buena) | `resultados` |
| Tiempo en T4 | ~1-2 min de entrenamiento | ~15-25 min |

`Config(debug=True)` ya ajusta lo necesario (2 epocas como maximo, `num_workers=0`, sin AMP,
sin freeze ni warmup), asi que no hay que recordarlo a mano.

Ejecuta primero en **debug**: si los pasos 6 y 7 terminan sin errores, cambia
`MODO_DEBUG = False` y vuelve a ejecutar desde esta celda para la corrida real.

In [ ]:
# Cambia a False para la corrida completa (25 epocas, batch 32, ~15-25 min en T4).
MODO_DEBUG = True

if MODO_DEBUG:
    cfg = Config(
        debug=True,                   # subconjunto + ajustes automaticos de Config
        debug_per_class=4,            # imagenes por clase en train/val/test
        epochs=2,
        batch_size=32,
        out_dir="resultados/debug",   # separado de los resultados definitivos
        device="auto",                # CUDA si esta disponible, si no CPU
    )
else:
    cfg = Config(
        debug=False,
        epochs=25,
        batch_size=32,
        backbone="efficientnet_b0",
        dropout=0.3,
        pretrained=True,
        lr=3e-4,                      # learning rate de la cabeza
        backbone_lr=3e-5,             # LR bajo del backbone al descongelar
        warmup_epochs=2,
        freeze_epochs=3,
        early_stop_patience=6,
        amp=True,                     # precision mixta (se ignora en CPU)
        out_dir="resultados",
        device="auto",
    )

cfg.ensure_dirs()                    # crea out_dir, figuras/, predicciones/ y data/
set_seed(cfg.seed)
device = resolve_device(cfg.device)

print("Modo         :", "DEBUG (subconjunto)" if MODO_DEBUG else "COMPLETO (dataset entero)")
print("Dispositivo  :", describe_device(device))
print("Epocas       : %d | batch: %d | img: %d" % (cfg.epochs, cfg.batch_size, cfg.img_size))
print("Backbone     : %s (pretrained=%s, dropout=%s)" % (cfg.backbone, cfg.pretrained, cfg.dropout))
print("Salida       :", Path(cfg.out_dir).resolve())
print("Checkpoint   :", cfg.checkpoint_path)
print("Metricas     :", cfg.metrics_path)
print("num_workers  :", cfg.num_workers, "(0 en debug; en Colab/Linux 2 es correcto)")
print("data_root    :", Path(cfg.data_root).resolve(), "(aqui se cachea Flowers-102)")

## Paso 6 - Entrenamiento

`src.train.train(cfg)` entrena y devuelve el historial; ademas escribe en `cfg.out_dir`:

- `best.pt` - checkpoint (pesos + `class_names` + config) de la mejor `val_acc`.
- `history.json` - perdida, accuracy, top-5, LR y mejor epoca por epoca.
- `config_usada.json` - hiperparametros exactos de la corrida.
- `figuras/curvas.png` - curvas de perdida, accuracy y learning rate.

La primera ejecucion descarga Flowers-102 (~330 MB) en `data/`; a partir de ahi se reutiliza
(y la ultima celda del notebook lo copia a Drive si esta montado).

**Donde entra Drive (importante).** Si en el paso 2 quedo montado `DRIVE_DIR`, esta celda
copia `resultados/` a Drive **justo despues de entrenar**: asi el checkpoint sobrevive a una
desconexion de Colab. La funcion `respaldar_en_drive()` queda a mano por si quieres llamarla
en otro momento. Con `USAR_DRIVE = False` todo se queda en la sesion.

**Reanudar tras una desconexion.** El contrato de `train(cfg)` no tiene parametro `resume`,
asi que para continuar: ejecuta los pasos 1-4, recupera `resultados/` desde
`Mi unidad/IA_flores` y sigue en el **paso 7** (evaluacion) o en el **paso 9** (demo) con el
`best.pt` recuperado. Para volver a entrenar, esta celda empieza de cero.

In [ ]:
import shutil
import time
import warnings
from pathlib import Path

from src.train import train


def respaldar_en_drive(origen="resultados", destino=None):
    # Copia la carpeta de resultados a Drive (si Drive esta montado).
    if DRIVE_DIR is None:
        print("Sin Drive montado: se omite el respaldo (modo local).")
        return None
    origen = Path(origen)
    if not origen.exists():
        print("Nada que respaldar todavia:", origen)
        return None
    destino = Path(destino) if destino is not None else DRIVE_DIR / "resultados"
    try:
        destino.mkdir(parents=True, exist_ok=True)
        shutil.copytree(origen, destino, dirs_exist_ok=True)
        print("Respaldo en Drive:", destino)
        return destino
    except Exception as exc:
        print("No se pudo respaldar en Drive:", exc)
        return None


# La descarga de pesos ImageNet se bloquea en entornos sin salida a esa URL: el
# contrato de src.model degrada solo a pretrained=False. Aqui se traduce el aviso
# crudo por uno mas legible (el entrenamiento sigue siendo valido).
with warnings.catch_warnings(record=True) as capturados:
    warnings.simplefilter("always")
    t0 = time.time()
    historial = train(cfg)             # DEBUG o COMPLETO segun MODO_DEBUG (paso 5)
    minutos = (time.time() - t0) / 60.0

for aviso in capturados:
    if "preentrenados" in str(aviso.message):
        print("\nNOTA: no se pudieron descargar los pesos ImageNet del backbone")
        print("      (red bloqueada). El contrato de src/model degrada solo a")
        print("      pretrained=False: se entrena desde cero y todo lo demas funciona.")
        print("      En Colab la descarga suele funcionar; si no, es lo mismo.")

print("\nEntrenamiento terminado en %.1f min" % minutos)
print("Mejor val_acc: %s (epoca %s)" % (historial.get("best_val_acc"), historial.get("best_epoch")))
print("Checkpoint   :", cfg.checkpoint_path, "| existe:", Path(cfg.checkpoint_path).is_file())

# Respaldo inmediato: si Colab se cae ahora, el checkpoint ya esta en Drive.
respaldar_en_drive("resultados")

# --- curvas del historial --------------------------------------------------
hist = load_json(cfg.history_path)
epocas = hist["epochs"]
fig, ejes = plt.subplots(1, 3, figsize=(16, 4))
ejes[0].plot(epocas, hist["train_loss"], marker="o", label="train")
ejes[0].plot(epocas, hist["val_loss"], marker="o", label="val")
ejes[0].set_xlabel("epoca"); ejes[0].set_ylabel("loss"); ejes[0].legend()
ejes[0].set_title("Perdida"); ejes[0].grid(alpha=0.3)
ejes[1].plot(epocas, hist["train_acc"], marker="o", label="train")
ejes[1].plot(epocas, hist["val_acc"], marker="o", label="val")
ejes[1].plot(epocas, hist["val_top5"], marker="s", alpha=0.6, label="val top-5")
ejes[1].set_xlabel("epoca"); ejes[1].set_ylabel("accuracy"); ejes[1].legend()
ejes[1].set_title("Accuracy (top-1 y top-5)"); ejes[1].grid(alpha=0.3)
ejes[2].plot(epocas, hist["lr"], marker="o", color="tab:green")
ejes[2].set_xlabel("epoca"); ejes[2].set_ylabel("learning rate")
ejes[2].set_title("LR de la cabeza (warmup + coseno)"); ejes[2].grid(alpha=0.3)
plt.tight_layout(); plt.show()

print("Historial:", cfg.history_path)
print("Figura de curvas:", cfg.figures_dir() / "curvas.png")

## Paso 7 - Evaluacion sobre el split oficial de test

`src.evaluate.evaluate(checkpoint_path, cfg, split="test")` carga el checkpoint, evalua el
split oficial de test y escribe:

- `metrics.json` con `split, num_images, top1, top5, macro_f1, per_class, per_class_f1,
  top_confusions, ruta_confusion, ruta_predicciones, segundos` (+ metadatos del checkpoint:
  `backbone`, `epoch`, `val_acc`, `img_size`, `num_classes`, `device`).
- `figuras/confusion_matrix.png` (matriz 102 x 102) y `figuras/top_confusions.png`.
- `predicciones/test_predicciones.csv` con columnas
  `ruta, etiqueta_real, prediccion, prob, correcto`.

La segunda celda imprime el resumen y el ranking de clases con peor F1, y muestra la matriz
de confusion. En modo debug el modelo solo ha visto un subconjunto y las metricas son malas
a proposito: valen para comprobar que el pipeline funciona, no para el informe.

In [ ]:
from src.evaluate import evaluate

metricas = evaluate(cfg.checkpoint_path, cfg, split="test")

print("\nClaves de metrics.json:", sorted(metricas.keys()))
print("top1 = %s | top5 = %s | macro_f1 = %s" % (
    metricas.get("top1"), metricas.get("top5"), metricas.get("macro_f1")))

In [ ]:
# --- 7.b Lectura de metrics.json, matriz de confusion y peores clases ------
import warnings

metricas = load_json(cfg.metrics_path)   # lo acaba de escribir evaluate()

# Resumen sin las estructuras grandes (per_class / per_class_f1 / top_confusions).
resumen = {k: v for k, v in metricas.items() if k not in ("per_class", "per_class_f1", "top_confusions")}
print(json.dumps(resumen, indent=2, ensure_ascii=False))
print("\nTop-1: %.2f%% | Top-5: %.2f%% | macro-F1: %.4f | imagenes: %s | %.1f s" % (
    100.0 * float(metricas["top1"]), 100.0 * float(metricas["top5"]),
    float(metricas["macro_f1"]), metricas["num_images"], float(metricas["segundos"])))

print("\nConfusiones mas frecuentes (real, pred, n):")
for fila in (metricas.get("top_confusions") or [])[:10]:
    real, pred, n = fila[0], fila[1], fila[2]
    print("  %-32s -> %-32s n=%s" % (real, pred, n))

# La matriz de confusion 102x102 la guarda evaluate() como PNG.
ruta_cm = Path(metricas.get("ruta_confusion") or (cfg.figures_dir() / "confusion_matrix.png"))
if ruta_cm.is_file():
    plt.figure(figsize=(9, 9))
    plt.imshow(plt.imread(ruta_cm))
    plt.axis("off")
    plt.title("Matriz de confusion (%dx%d) - split %s" % (
        cfg.num_classes, cfg.num_classes, metricas.get("split", "test")))
    with warnings.catch_warnings():
        # Si un modulo del proyecto forzo el backend "Agg" (lo hace src.evaluate
        # para no abrir ventanas), plt.show() avisa de que no puede dibujar en
        # pantalla. La figura se genera igual y se incrusta en el notebook.
        warnings.filterwarnings("ignore", message=".*non-interactive.*")
        plt.show()
    print("Figura:", ruta_cm)
else:
    print("No encuentro la matriz de confusion en", ruta_cm)

# Peores clases por F1 (especies que mas se confunden).
per_class_f1 = metricas.get("per_class_f1")
if isinstance(per_class_f1, dict) and per_class_f1:
    try:
        peores = sorted(per_class_f1.items(), key=lambda kv: float(kv[1]))[:5]
        print("\nClases con peor F1 (especies que mas se confunden):")
        for nombre, f1 in peores:
            print("  %-40s F1=%.3f" % (nombre, float(f1)))
    except (TypeError, ValueError):
        print("\nper_class_f1 no tiene el formato nombre -> f1; se omite el ranking.")
else:
    print("\nmetrics.json no trae per_class_f1 como diccionario; se omite el ranking.")

## Paso 8 - Resumen listo para pegar en el informe

Esta celda imprime un bloque en **Markdown** pensado para copiarse tal cual en
`INFORME.md`, seccion **8. Resultados**, sustituyendo los marcadores
`_pendiente de la corrida en Colab_`:

- Tabla con top-1, top-5, macro-F1, numero de imagenes, segundos de evaluacion,
  mejor epoca, mejor `val_acc` y numero de parametros.
- Tabla con las 10 confusiones mas frecuentes (`top_confusions` de `metrics.json`), que
  corresponde a la seccion **9.2** del informe.

Copia el bloque que aparece entre las lineas `---8<---` y `--->8---` (o selecciona el texto
en la salida de la celda y pegalo en el informe). Las cifras salen **directamente** de
`resultados/metrics.json` y `resultados/history.json`: no hay ningun valor a mano. En modo
debug el bloque usa las rutas reales de esa corrida (`resultados/debug/...`) y anade un aviso
recordando que hay que repetir la corrida completa antes de entregarlo.

In [ ]:
# --- 8.a Resumen pegable en INFORME.md (seccion 8) --------------------------
from pathlib import Path

metricas = load_json(cfg.metrics_path)
hist = load_json(cfg.history_path)


def _pct(valor):
    try:
        return "%.2f %%" % (100.0 * float(valor))
    except (TypeError, ValueError):
        return "s/d"


def _segundos(valor):
    try:
        total = float(valor)
    except (TypeError, ValueError):
        return "s/d"
    return "%.1f s (%.1f min)" % (total, total / 60.0)


def _entero(valor):
    try:
        return f"{int(valor):,}".replace(",", " ")
    except (TypeError, ValueError):
        return "s/d"


lineas = []
lineas.append("<!-- Copiado del notebook de Colab (paso 8) -->")
lineas.append("")
lineas.append("**Corrida:** %s | dispositivo: %s | backbone: %s" % (
    "DEBUG (subconjunto)" if MODO_DEBUG else "COMPLETA (1020/1020/6149)",
    metricas.get("device", "s/d"),
    metricas.get("backbone", cfg.backbone)))
lineas.append("")
lineas.append("### 8.1 Metricas principales (split oficial de test)")
lineas.append("")
lineas.append("| Metrica | Valor | Fuente del dato |")
lineas.append("|---|---|---|")
origen = Path(cfg.out_dir).as_posix()
lineas.append("| Exactitud top-1 | %s | `%s/metrics.json` -> `top1` |" % (_pct(metricas.get("top1")), origen))
lineas.append("| Exactitud top-5 | %s | `%s/metrics.json` -> `top5` |" % (_pct(metricas.get("top5")), origen))
lineas.append("| Macro-F1 | %.4f | `%s/metrics.json` -> `macro_f1` |" % (float(metricas.get("macro_f1", 0.0)), origen))
lineas.append("| Imagenes evaluadas | %s | `%s/metrics.json` -> `num_images` |" % (_entero(metricas.get("num_images")), origen))
lineas.append("| Tiempo de evaluacion | %s | `%s/metrics.json` -> `segundos` |" % (_segundos(metricas.get("segundos")), origen))
lineas.append("| Tiempo de entrenamiento | %s | `%s/history.json` -> `segundos` |" % (_segundos(hist.get("segundos")), origen))
lineas.append("| Mejor epoca | %s | `%s/history.json` -> `best_epoch` |" % (_entero(hist.get("best_epoch")), origen))
lineas.append("| Epocas ejecutadas | %s | `%s/history.json` -> `epochs_run` |" % (_entero(hist.get("epochs_run")), origen))
lineas.append("| Mejor exactitud de validacion | %s | `%s/history.json` -> `best_val_acc` |" % (_pct(hist.get("best_val_acc")), origen))
lineas.append("| Dispositivo utilizado | %s | `%s/history.json` -> `device` |" % (str(hist.get("device", "s/d")), origen))
lineas.append("| Numero de parametros | %s | `%s/history.json` -> `num_params` |" % (_entero(hist.get("num_params")), origen))
lineas.append("")
lineas.append("### 9.2 Confusiones mas frecuentes (real -> predicha)")
lineas.append("")
lineas.append("> Bloque para la seccion 9.2 del informe (analisis de errores).")
lineas.append("")
lineas.append("| # | Especie real | Especie predicha | Imagenes |")
lineas.append("|---|---|---|---:|")
confusiones = list(metricas.get("top_confusions") or [])[:10]
if confusiones:
    for posicion, fila in enumerate(confusiones, start=1):
        lineas.append("| %d | %s | %s | %s |" % (posicion, fila[0], fila[1], fila[2]))
else:
    lineas.append("| - | (sin confusiones registradas) | - | 0 |")
lineas.append("")
lineas.append("**Figuras:** `%s` (curvas), `%s` (matriz de confusion), `%s` (top confusiones)." % (
    cfg.figures_dir() / "curvas.png",
    cfg.figures_dir() / "confusion_matrix.png",
    cfg.figures_dir() / "top_confusions.png"))

if MODO_DEBUG:
    lineas.append("")
    lineas.append("> AVISO: estas cifras son del MODO DEBUG (subconjunto). Para el informe hay")
    lineas.append("> que repetir la corrida con `MODO_DEBUG = False` y volver a copiar el bloque.")

bloque = "\n".join(lineas)
print("-----------------------------8<-----------------------------")
print(bloque)
print("----------------------------->8-----------------------------")

# Se guarda tambien en disco para no perderlo.
ruta_resumen = Path(cfg.out_dir) / "resumen_informe.md"
ruta_resumen.write_text(bloque + "\n", encoding="utf-8")
print("\nResumen guardado en:", ruta_resumen)

## Paso 9 - Prediccion suelta y demo interactivo (Gradio)

**Prediccion suelta.** `src.predict.predict_image(imagen, checkpoint, topk)` clasifica una
imagen y devuelve pares `(especie, probabilidad)` decrecientes. Acepta ruta, `PIL.Image`,
`numpy.ndarray` o la tupla de Gradio, y cachea el modelo para no recargarlo en cada clic.

**Demo.** `src.app.build_app(checkpoint_path, topk)` construye la interfaz: subes una imagen
(o usas la webcam) y devuelve el top-k de especies con su probabilidad. Si el checkpoint no
existe, la app **no se cae**: muestra el aviso "entrena primero".

**`share=True`.** Con `share=False` la demo solo se ve dentro de Colab. Con `share=True`
Gradio crea un enlace publico temporal (`https://xxxx.gradio.live`, caduca en ~72 h) que
puedes abrir desde el movil: muy util para ensenar la demo. Cualquiera que tenga el enlace
puede usarla mientras este viva, asi que no la dejes abierta con imagenes privadas.

La ultima celda de esta seccion levanta la demo en un segundo plano
(`prevent_thread_lock=True`) y **devuelve el control** para que puedas seguir con el paso 10
sin pararla a mano. Si prefieres el comportamiento clasico (la celda se queda ocupada con el
iframe), pon `BLOQUEAR_CELDA = True`.

In [ ]:
from src.predict import predict_image

# Imagen de ejemplo: la primera disponible en data/ (o assets/ si no hay dataset).
candidatas = sorted(Path("data").rglob("*.jpg")) if Path("data").is_dir() else []
if not candidatas:
    candidatas = sorted(Path("assets").rglob("*.jpg"))

# min_conf=0.0 -> se muestra SIEMPRE el top-k. Con el umbral por defecto (0.25) un
# modelo poco entrenado (por ejemplo en MODO_DEBUG) no llega ni al 3 % de confianza,
# porque el softmax queda muy plano, y esta celda imprimiria solo "no reconocida".
MIN_CONF = 0.0

if candidatas and Path(cfg.checkpoint_path).is_file():
    imagen_ejemplo = candidatas[0]
    print("Imagen de ejemplo:", imagen_ejemplo)
    print("Checkpoint       :", cfg.checkpoint_path)
    resultados = predict_image(
        imagen_ejemplo, cfg.checkpoint_path, topk=cfg.topk, min_conf=MIN_CONF
    )
    for posicion, (nombre, probabilidad) in enumerate(resultados, start=1):
        print("%2d. %-42s %6.2f%%" % (posicion, nombre, 100.0 * probabilidad))
    maxima = max((p for _, p in resultados), default=0.0)
    if maxima < 0.25:
        print()
        print("NOTA: la confianza maxima es %.2f %%, por debajo del 25 %%." % (100.0 * maxima))
        print("      Es normal si entrenaste en MODO_DEBUG o sin pesos preentrenados:")
        print("      el softmax queda plano. Con la corrida completa (MODO_DEBUG=False)")
        print("      la confianza sube y la especie propuesta es mas fiable.")
else:
    print("Sin imagen de ejemplo o sin checkpoint: se omite la prediccion suelta.")


In [ ]:
from pathlib import Path

from src.app import build_app

CHECKPOINT = str(cfg.checkpoint_path)   # resultados/best.pt (o resultados/debug/best.pt)
print("Checkpoint de la demo:", CHECKPOINT, "| existe:", Path(CHECKPOINT).is_file())

# Umbral por debajo del cual la demo anade un AVISO de "confianza baja".
# La demo SIEMPRE muestra el top-k con las especies; este umbral ya no borra la
# prediccion (antes respondia "no reconocida" incluso acertando la especie).
MIN_CONF = 0.25

demo = build_app(checkpoint_path=CHECKPOINT, topk=cfg.topk, min_conf=MIN_CONF)
print("Interfaz construida:", type(demo).__name__, "| tiene launch:", hasattr(demo, "launch"))

# share=False -> solo el iframe dentro de Colab (privado y mas rapido).
# share=True  -> enlace publico temporal (https://xxxx.gradio.live, ~72 h),
#                para abrirlo desde el movil o ensenar la demo a alguien.
USAR_SHARE = False

# prevent_thread_lock=True lanza el servidor en segundo plano y devuelve el control:
# asi la demo queda viva (iframe/enlace) pero la ejecucion no se queda bloqueada.
BLOQUEAR_CELDA = False

resultado = demo.launch(share=USAR_SHARE, debug=False, prevent_thread_lock=not BLOQUEAR_CELDA)

url_local = None
url_publica = None
if isinstance(resultado, tuple) and len(resultado) >= 3:
    url_local, url_publica = resultado[1], resultado[2]
elif isinstance(resultado, str):
    url_local = resultado

print("Demo lanzada.")
if url_local:
    print("  URL local    :", url_local)
if USAR_SHARE and url_publica:
    print("  URL publica  :", url_publica)
elif USAR_SHARE:
    print("  AVISO: se pidio share=True pero Gradio no devolvio enlace publico.")
print("  Para detenerla: ejecuta la celda de abajo (demo.close()).")
print("  Si falta el checkpoint, la interfaz muestra el aviso de entrenar primero.")


### Celda auxiliar: parar la demo

La demo del paso 9 se queda escuchando en un puerto local (`http://127.0.0.1:7860` por
defecto) para que el iframe o el enlace publico sigan funcionando. Ejecuta esta celda solo si
quieres **liberar el puerto** (por ejemplo, para volver a lanzarla cambiando `share` o el
checkpoint). Si nunca la ejecutas, no pasa nada: el servidor muere con la sesion de Colab.

In [ ]:
# --- Celda auxiliar OPCIONAL: parar la demo de Gradio ----------------------
# Ejecutala solo si quieres liberar el puerto (por ejemplo antes de relanzarla).
try:
    demo.close()
    print("Demo detenida.")
except NameError:
    print("No hay ninguna demo levantada en esta sesion.")
except Exception as exc:
    print("No se pudo cerrar la demo:", exc)

## Paso 10 - Empaquetar resultados, descargarlos y cachear `data/` en Drive

Comprime la carpeta `resultados/` (checkpoint, historial, `metrics.json`, `config_usada.json`,
predicciones, figuras y el `resumen_informe.md`) en `resultados_flores.zip`, lo descarga en tu
PC y guarda una copia en Drive.

Ademas copia `data/` a Drive para que la proxima sesion de Colab **no vuelva a descargar los
330 MB**: el paso 2 lo restaura automaticamente. Si no hay Drive, todo esto se omite sin
errores.

Estos artefactos no se versionan en Git: son resultados generados (ver `.gitignore`).

In [ ]:
import shutil
import zipfile
from pathlib import Path

RAIZ_RESULTADOS = Path("resultados")            # incluye resultados/debug si existe
NOMBRE_ZIP = (Path("/content") if Path("/content").is_dir() else Path(".")) / "resultados_flores.zip"

if not RAIZ_RESULTADOS.exists():
    raise FileNotFoundError("No hay resultados/: ejecuta antes el paso 6 (entrenamiento).")

with zipfile.ZipFile(NOMBRE_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for ruta in sorted(RAIZ_RESULTADOS.rglob("*")):
        if ruta.is_file():
            zf.write(ruta, ruta.as_posix())

print("ZIP creado: %s (%.1f MB)" % (NOMBRE_ZIP, NOMBRE_ZIP.stat().st_size / 1024 ** 2))
with zipfile.ZipFile(NOMBRE_ZIP) as zf:
    for nombre in zf.namelist()[:20]:
        print("   ", nombre)

# Descarga al PC (solo funciona en Colab) y copia a Drive para no perderlo.
try:
    from google.colab import files

    files.download(str(NOMBRE_ZIP))
except ImportError:
    print("Fuera de Colab: el ZIP esta en", NOMBRE_ZIP)
except Exception as exc:
    print("No se pudo lanzar la descarga:", exc)

if DRIVE_DIR is not None:
    try:
        shutil.copy2(NOMBRE_ZIP, DRIVE_DIR / NOMBRE_ZIP.name)
        print("Copia del ZIP en Drive:", DRIVE_DIR / NOMBRE_ZIP.name)
    except Exception as exc:
        print("No se pudo copiar el ZIP a Drive:", exc)

# --- cache del dataset en Drive (evita re-descargar 330 MB la proxima vez) --
if DRIVE_DIR is None:
    print("\nCache del dataset: sin Drive, se omite (el dataset se queda en la sesion).")
elif not Path("data").is_dir():
    print("\nCache del dataset: no hay data/ que copiar.")
else:
    try:
        destino_data = DRIVE_DIR / "data"
        print("\nCopiando data/ a Drive (330 MB, puede tardar un par de minutos)...")
        shutil.copytree("data", destino_data, dirs_exist_ok=True)
        print("Dataset cacheado en:", destino_data)
        print("La proxima sesion lo restaurara en el paso 2 y no volvera a descargarlo.")
    except Exception as exc:
        print("No se pudo cachear data/ en Drive:", exc)
        print("El notebook termina igual; la proxima sesion volvera a descargar el dataset.")

# --- inventario final de artefactos ----------------------------------------
print("\n=== Artefactos de la corrida ===")
for etiqueta, ruta in [
    ("checkpoint", cfg.checkpoint_path),
    ("historial", cfg.history_path),
    ("config usada", cfg.config_path),
    ("metricas", cfg.metrics_path),
    ("resumen informe", Path(cfg.out_dir) / "resumen_informe.md"),
    ("curvas", cfg.figures_dir() / "curvas.png"),
    ("matriz de confusion", cfg.figures_dir() / "confusion_matrix.png"),
    ("top confusiones", cfg.figures_dir() / "top_confusions.png"),
    ("predicciones", cfg.predictions_dir() / "test_predicciones.csv"),
    ("ZIP de resultados", NOMBRE_ZIP),
]:
    print("  %-18s %s  %s" % (etiqueta, "OK " if Path(ruta).exists() else "FALTA", ruta))

## Checklist final

**Antes de ejecutar**

- [ ] `Entorno de ejecucion > Cambiar tipo de entorno de ejecucion > T4 GPU` (y "Guardar").
- [ ] Paso 1: `torch.cuda.is_available()` -> `True` y la GPU aparece como `Tesla T4`.
- [ ] Paso 1: **no** se ejecuto `pip install -r requirements.txt`; solo entran `gradio`,
      `scikit-learn`, `matplotlib` y `scipy` si faltaban. `torch` nunca se toca.
- [ ] Paso 2: Drive montado si quieres que el checkpoint y el dataset sobrevivan
      (`USAR_DRIVE = True`).
- [ ] Paso 3: `MODO_CODIGO = "auto"` (detecta el proyecto local o lo clona de GitHub) y
      `src/config.py` encontrado en la raiz.

**Rutas que deben existir**

- [ ] `src/config.py`, `src/data.py`, `src/model.py`, `src/train.py`, `src/evaluate.py`,
      `src/predict.py`, `src/app.py` en la raiz del proyecto.
- [ ] `data/` con Flowers-102 (1020 train / 1020 val / 6149 test; ~330 MB).
- [ ] `resultados/best.pt` tras el paso 6 y `resultados/metrics.json` tras el paso 7
      (en modo debug, `resultados/debug/best.pt` y `resultados/debug/metrics.json`).

**Orden recomendado**

1. **Primero en debug** (`MODO_DEBUG = True`, ya viene asi): pasos 1 a 10 seguidos.
   Tarda **~1-2 min** de entrenamiento y confirma que la instalacion, la descarga del
   dataset, el modelo, la evaluacion y la demo funcionan.
2. **Luego la corrida completa**: pon `MODO_DEBUG = False` en el paso 5 y ejecuta otra vez
   desde esa celda hasta el final. Son **25 epocas, batch 32, ~15-25 min en T4**. El modo
   debug escribe en `resultados/debug`, asi que **no pisa** la corrida buena.

**Donde queda el checkpoint**

| Corrida | Checkpoint | Metricas | Figuras | Predicciones |
|---|---|---|---|---|
| Debug | `resultados/debug/best.pt` | `resultados/debug/metrics.json` | `resultados/debug/figuras/` | `resultados/debug/predicciones/` |
| Completa | `resultados/best.pt` | `resultados/metrics.json` | `resultados/figuras/` | `resultados/predicciones/` |

El checkpoint guarda pesos + `class_names` + config; para evaluarlo o usarlo en la demo basta
con apuntar a esa ruta.

**Como reanudar tras una desconexion de Colab**

1. Vuelve a ejecutar los pasos 1-4 (GPU, Drive, dependencias, codigo).
2. Recupera los resultados desde Drive a la raiz del proyecto:
   `shutil.copytree(DRIVE_DIR / "resultados", "resultados", dirs_exist_ok=True)`.
3. Salta al **paso 7** (evaluacion) o al **paso 9** (demo) con el `best.pt` recuperado.
4. Si perdiste el entrenamiento antes de guardar: `train(cfg)` no tiene `resume`, hay que
   volver a entrenar desde el paso 6 (el dataset ya esta cacheado, no se re-descarga).

**Como descargar los resultados**

- El paso 10 crea `resultados_flores.zip` y lo descarga (`files.download`) en Colab; tambien
  deja copia en `Mi unidad/IA_flores/`.
- Si la descarga automatica falla: abre el panel de archivos de Colab (icono de carpeta),
  navega a `resultados_flores.zip` y usa el menu "Descargar".
- Para el informe: pega el bloque del **paso 8** en `INFORME.md` seccion 8 y copia
  `resultados/figuras/*.png` a `resultados/figuras/` del repositorio.

**Tiempos estimados en T4**

| Escenario | Tiempo |
|---|---|
| Descarga de Flowers-102 (~330 MB) | 1-3 min (solo la primera vez) |
| Pasos 5-7 en modo debug | ~2-4 min |
| Corrida completa (25 epocas, batch 32) | ~15-25 min |
| Paso 9 (demo) | inmediato: solo carga el checkpoint |
| Paso 10 (ZIP + cache en Drive) | ~1-3 min |

Si algo falla: ejecuta primero en modo debug y lee el error literal antes de tocar
hiperparametros.